# TikTok Data Collection

Collects the caption and comments (with replies) of the TikTok posts listed in `tiktok_post_links.txt`.

No TikTok account or login is needed: the data is read from TikTok's public web pages and the same comment endpoints the website uses.

Steps:
1. Put one post link per line in `tiktok_post_links.txt` (e.g. `https://www.tiktok.com/@user/video/1234567890123456789`, a `.../photo/...` link, or a short link like `https://vm.tiktok.com/XXXXXXX/`).
2. Run the cells in order.

Each post is saved as soon as it is collected. Posts already in `TikTokComments.csv` are skipped on later runs, so only new links and links that failed before are collected.

In [ ]:
import itertools
import json
import os
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Configuration

In [ ]:
LINKS_FILE = "tiktok_post_links.txt"
OUTPUT_CSV = "TikTokComments.csv"
MAX_COMMENTS_PER_POST = 500  # top-level comments per post; replies of those comments are also collected
DELAY_BETWEEN_POSTS = 10  # seconds, to avoid being rate limited by TikTok
DELAY_BETWEEN_PAGES = 1  # seconds
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

## Read post links

In [ ]:
def read_post_links(path):
    # One link per line; blank lines and lines starting with # are ignored
    with open(path, 'r', encoding='utf-8') as links_file:
        lines = [line.strip() for line in links_file]
    return [line for line in lines if line and not line.startswith('#')]

def read_saved_texts(path):
    # Posts already in the output file are skipped
    if not os.path.exists(path):
        return pd.DataFrame(columns=['post_url'])
    return pd.read_csv(path, index_col=0, encoding='utf-8')

post_links = read_post_links(LINKS_FILE)
saved_texts = read_saved_texts(OUTPUT_CSV)
saved_links = set(saved_texts['post_url'])
links_to_collect = [link for link in post_links if link not in saved_links]
print(len(post_links), 'links,', len(post_links) - len(links_to_collect), 'already collected,', len(links_to_collect), 'to collect')

## Get captions and comments

In [ ]:
COMMENTS_URL = 'https://www.tiktok.com/api/comment/list/'
REPLIES_URL = 'https://www.tiktok.com/api/comment/list/reply/'
PAGE_SIZE = 50  # the most comments TikTok returns per request

def make_session():
    # Retries dropped connections and rate limit / server errors, waiting longer after each attempt
    retry = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    session = requests.Session()
    session.mount('https://', HTTPAdapter(max_retries=retry))
    session.headers.update({'User-Agent': USER_AGENT, 'Referer': 'https://www.tiktok.com/'})
    return session

def get_post_item(session, url):
    # The video page embeds the post data (caption, author, date, likes) as JSON; short links are redirected
    response = session.get(url, timeout=30)
    response.raise_for_status()
    script = BeautifulSoup(response.text, 'html.parser').find('script', id='__UNIVERSAL_DATA_FOR_REHYDRATION__')
    if script is None:
        raise ValueError('No post data in the page (blocked or not a post link): ' + url)
    video_detail = json.loads(script.string)['__DEFAULT_SCOPE__'].get('webapp.video-detail', {})
    if video_detail.get('statusCode') != 0:
        raise ValueError('Post not available (deleted, private or not a post link): ' + url)
    return video_detail['itemInfo']['itemStruct']

def get_json(session, url, params):
    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    page = response.json()
    if page.get('status_code') != 0:
        raise ValueError('TikTok error: ' + str(page.get('status_msg')))
    return page

def iter_pages(session, url, params):
    cursor = 0
    while True:
        page = get_json(session, url, {**params, 'aid': '1988', 'count': PAGE_SIZE, 'cursor': cursor})
        yield from page.get('comments') or []
        if not page.get('has_more'):
            return
        cursor = page['cursor']
        time.sleep(DELAY_BETWEEN_PAGES)

def iter_comment_nodes(session, video_id):
    return iter_pages(session, COMMENTS_URL, {'aweme_id': video_id})

def iter_reply_nodes(session, video_id, comment_node):
    if comment_node.get('reply_comment_total', 0) == 0:
        return iter([])
    return iter_pages(session, REPLIES_URL, {'item_id': video_id, 'comment_id': comment_node['cid']})

def text_row(url, video_id, source, username, created_at, likes, text):
    return {'post_url': url, 'video_id': video_id, 'source': source, 'username': username,
            'created_at': created_at, 'likes': likes, 'text': text}

def comment_row(url, video_id, source, node):
    created_at = pd.to_datetime(node['create_time'], unit='s')
    return text_row(url, video_id, source, node['user'].get('unique_id'), created_at,
                    node.get('digg_count', 0), node['text'])

def get_post_texts(session, url, max_comments):
    item = get_post_item(session, url)
    video_id = item['id']
    rows = []
    if item.get('desc'):
        created_at = pd.to_datetime(int(item['createTime']), unit='s')
        rows.append(text_row(url, video_id, 'caption', item['author']['uniqueId'], created_at,
                             item['stats']['diggCount'], item['desc']))
    for comment in itertools.islice(iter_comment_nodes(session, video_id), max_comments):
        rows.append(comment_row(url, video_id, 'comment', comment))
        for reply in iter_reply_nodes(session, video_id, comment):
            rows.append(comment_row(url, video_id, 'reply', reply))
    return rows

## Save texts

In [ ]:
def clean_text(text):
    # same cleaning as the other notebooks: remove #, commas, new lines and @mentions
    text = text.replace('#', '').replace(',', ' ').replace('\n', ' ')
    return re.sub(r'@[^\s]+', '', text).strip()

def clean_rows(rows):
    cleaned = [{**row, 'text': clean_text(row['text'])} for row in rows]
    return [row for row in cleaned if row['text']]

def append_rows(path, rows, first_index):
    if not rows:
        return
    table = pd.DataFrame(rows, index=range(first_index, first_index + len(rows)))
    table.to_csv(path, mode='a', header=not os.path.exists(path), encoding='utf-8')

## Collect the posts

In [ ]:
session = make_session()
next_index = int(saved_texts.index.max()) + 1 if len(saved_texts) else 0
failed_links = []
for i, url in enumerate(links_to_collect):
    print(i, '--------------- getting comments of', url, '------------------')
    try:
        post_texts = clean_rows(get_post_texts(session, url, MAX_COMMENTS_PER_POST))
        append_rows(OUTPUT_CSV, post_texts, next_index)
        next_index += len(post_texts)
        print('got', len(post_texts), 'texts')
    except (requests.RequestException, ValueError) as err:
        print('failed:', err)
        failed_links.append(url)
    time.sleep(DELAY_BETWEEN_POSTS)

print('failed links:', len(failed_links))

In [ ]:
tiktok_df = pd.read_csv(OUTPUT_CSV, index_col=0, encoding='utf-8')
print(len(tiktok_df), 'texts in', OUTPUT_CSV)
tiktok_df.head(900)